# 4 · XFoil Aerodynamic Polar Generation

**Author:** Héctor Fernández Pinacho  
**Lab:** IDEAL Lab · ETH Zürich

For every propeller in the design space, computes the aerodynamic input coefficients QProp needs at each blade station by running XFoil (a 2-D coupled panel / boundary-layer solver) across a geometry-adaptive Reynolds-number grid, then fitting the lift curve and the drag polar of every converged run. Each propeller carries 7 aerodynamic stations (the hub station at 8.25 mm plus s1–s6 at r/R fractions 0.20–0.92); each station is assigned the best available polar through a documented tier hierarchy, and every configuration receives a thrust-weighted confidence score. The same flow is then repeated for the 10 recovered validation propellers, producing the `val_`-prefixed outputs.

**Physics inputs:** `csv/01_geometry.csv`, `csv/03_naca_codes.csv`, `utils/xfoil.exe` (and, for the validation subset, `utils/00_validation_geometry.csv` and `csv/val_03_naca_codes.csv`)

**Physics outputs:** `csv/04_xfoil_polars.csv` (per-station QProp coefficients CL0, CL_a, CLmin, CLmax, CD0, CD2u, CD2l, CLCD0, REref, REexp, plus geometry, tier and diagnostics), `csv/04_xfoil_failed_configs.csv`, and the validation-subset equivalents `csv/val_04_xfoil_polars.csv`, `csv/val_04_xfoil_failed_configs.csv`

**Structure:**

1. Imports
2. Configuration — all tunable constants, paths and settings
3. Function definitions — cache management, XFoil execution, polar parsing and fitting, REexp fitting, polar assignment, confidence scoring
4. Main code — station table, Re grid, XFoil sweep, REexp fits, polar assembly and export, validation subset, quality report


## 1. Imports

In [ ]:
import os
import sys

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True

import subprocess
import tempfile
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from tqdm.auto import tqdm

base_dir = Path(globals().get('__vsc_ipynb_file__', Path.cwd())).parent
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))
import pipeline_config as cfg
from utils import measurements

## 2. Configuration

In [ ]:
csv_dir = base_dir / 'csv'
polar_dir_main = base_dir / 'xfoil_polars'
validation_polar_dir = base_dir / 'xfoil_polars' / 'validation'
xfoil_exe = base_dir / 'utils' / 'xfoil.exe'

csv_dir.mkdir(parents=True, exist_ok=True)
polar_dir_main.mkdir(parents=True, exist_ok=True)
validation_polar_dir.mkdir(parents=True, exist_ok=True)

geometry_csv = csv_dir / cfg.CSV_NAMES['geometry']
naca_csv = csv_dir / cfg.CSV_NAMES['naca_codes']
output_csv = csv_dir / cfg.CSV_NAMES['xfoil_polars']
failed_csv_path = csv_dir / cfg.CSV_NAMES['xfoil_failed']

inner_anchor_radius_mm = cfg.INNER_ANCHOR_RADIUS_MM
hub_station_radius_mm = cfg.HUB_STATION_RADIUS_MM
qprop_station_fractions = cfg.QPROP_STATION_FRACTIONS
qprop_station_names = cfg.QPROP_STATION_NAMES
all_station_names = ['hub'] + qprop_station_names

launch_rpm = int(cfg.LAUNCH_RPM)
max_rpm = cfg.NB4_MAX_RPM
floor_rpm = cfg.NB4_FLOOR_RPM

launch_omega = launch_rpm * 2 * np.pi / 60.0
air_kinematic_viscosity = cfg.AIR_KINEMATIC_VISCOSITY

re_floor = cfg.RE_FLOOR
re_ceiling = cfg.RE_CEILING
re_rounding_step = cfg.RE_ROUNDING_STEP
re_samples_per_decade = cfg.RE_SAMPLES_PER_DECADE

ncrit = cfg.NCRIT
xtr_outer = cfg.XTR_OUTER
xtr_hub = cfg.XTR_HUB
hub_re_threshold = cfg.HUB_RE_THRESHOLD

alpha_start = cfg.ALPHA_START
alpha_end = cfg.ALPHA_END
alpha_step = cfg.ALPHA_STEP
xfoil_max_iterations = cfg.XFOIL_MAX_ITERATIONS
xfoil_timeout_sec = cfg.XFOIL_TIMEOUT_SEC
num_parallel_workers = min(os.cpu_count() or 4, cfg.XFOIL_MAX_WORKERS_CAP)

fit_window_alpha_low = cfg.XFOIL_FIT_ALPHA_LOW
fit_window_alpha_high = cfg.XFOIL_FIT_ALPHA_HIGH
min_points_for_fit = cfg.XFOIL_MIN_POINTS_FIT

cla_minimum = cfg.CLA_MINIMUM
cla_maximum = cfg.CLA_MAXIMUM
cd0_minimum = cfg.CD0_MINIMUM
cd0_maximum = cfg.CD0_MAXIMUM
cd0_reference_cl = cfg.CD0_REFERENCE_CL
stall_safety_buffer = cfg.STALL_SAFETY_BUFFER
cl_jump_threshold = cfg.CL_JUMP_THRESHOLD
min_valid_polar_rows = cfg.MIN_VALID_POLAR_ROWS
min_valid_file_bytes = cfg.MIN_VALID_FILE_BYTES

re_exponent_r2_gate = cfg.RE_EXPONENT_R2_GATE
qprop_aero_keys = cfg.QPROP_AERO_KEYS

print('Configuration loaded.')
print(f'  Launch RPM       : {launch_rpm}  →  RPM envelope {floor_rpm}–{max_rpm}')
print(f'  Re floor/ceiling : {re_floor:,} / {re_ceiling:,}')
print(f'  Ncrit            : {ncrit}  (indoor enclosed environment)')
print(f'  xtr outer        : {xtr_outer}  (surface roughness, Re > {hub_re_threshold:,})')
print(f'  xtr hub          : {xtr_hub}  (near-LE separation, Re <= {hub_re_threshold:,})')
print(f'  Alpha range      : {alpha_start}° to {alpha_end}°  step {alpha_step}°')
print(f'  REexp R² gate    : {re_exponent_r2_gate}  (fall back to -0.5 if below)')
print(f'  Parallel workers : {num_parallel_workers}')

## 3. Function Definitions

**Cache and station geometry**

- **make_polar_cache_filename(naca_code, reynolds_number, transition_location)** — builds the unique cache filename that encodes the NACA code, Reynolds number, forced-transition location and Ncrit of a run, so identical runs are never repeated and changed settings never reuse a stale file.
- **get_polar_cache_path(polar_dir, naca_code, reynolds_number, transition_location)** — returns the full path of that cached polar inside the given cache folder.
- **polar_cache_is_valid(cache_path)** — True when a cached file exists and is large enough to be a real polar rather than a failed stub.
- **get_transition_location_for_re(reynolds_number)** — picks the forced laminar-to-turbulent transition point: `xtr_hub` at hub-level Reynolds numbers, `xtr_outer` above.
- **build_chord_or_twist_spline(r_inner, r_mid, r_outer, value_inner, value_mid, value_outer)** — natural cubic spline through the three blade anchor stations, the same spline model Rhino uses for the surface loft, so chord or twist can be read at any radius.
- **compute_reference_re(radius_mm, chord_mm, angular_velocity)** — Reynolds number a blade element sees at a given radius, chord and rotation speed, clamped to the grid bounds and rounded for cache reuse.
- **make_station_row(config_id, station_name, radius_mm, naca_code, chord_spline, twist_spline)** — evaluates chord and twist at one station radius and packages the station dictionary row.
- **build_station_table(propeller_table)** — places the 7 stations (hub + s1–s6) for every propeller and returns the full station table.
- **derive_re_sample_grid(station_table)** — derives the set of Re values at which XFoil is actually run: it spans from the hard XFoil floor up to the highest Re reached in the dataset at the overspeed RPM, with logarithmically spaced points per decade rounded for cache reuse. Returns the grid and its coverage statistics.
- **build_job_list(naca_codes, re_samples)** — the full XFoil job list: every unique NACA code at every Re sample level.

**XFoil execution**

- **build_xfoil_batch_script(naca_code, polar_output_file, reynolds_number, transition_location)** — writes the XFoil command sequence for one run: `PLOP G F` (headless), `NACA` (load airfoil), `OPER`/`ITER`/`VISC` (viscous mode at the given Re), `VPAR N`/`XTR` (Ncrit and forced transition), `PACC` + `ASEQ` (accumulate the alpha sweep to the polar file), `QUIT`.
- **run_one_xfoil_job(polar_dir, naca_code, reynolds_number, transition_location)** — runs XFoil once for a single case with cache check and timeout handling; a converged polar is moved into the cache. Returns True on success.
- **run_xfoil_jobs_in_parallel(jobs, polar_dir, label)** — runs all uncached jobs across CPU workers with a progress bar.

**Polar parsing and coefficient fitting**

- **read_xfoil_polar_file(file_path)** — loads the alpha, CL, CD columns from one XFoil output file.
- **remove_convergence_glitches(alpha_values, cl_values, cd_values)** — sorts by angle and drops points where the lift jumps unphysically between neighbouring angles (non-converged points).
- **get_attached_flow_mask(alpha_values)** — selects the clean attached-flow part of the polar to fit on, with a slightly widened fallback window.
- **fit_lift_curve(alpha_in_fit_window, cl_in_fit_window)** — fits the straight lift line and returns CL0 and the lift-curve slope in 1/rad.
- **fit_drag_polar(cl_in_fit_window, cd_in_fit_window)** — fits the parabolic drag bucket and returns the minimum drag, the CL at minimum drag, and the symmetric curvature.
- **fit_asymmetric_drag_curvatures(cl_in_fit_window, cd_in_fit_window, cl_at_min_drag, symmetric_curvature)** — fits the drag curvature separately above (CD2u) and below (CD2l) the minimum-drag CL, because a cambered airfoil's drag bucket is asymmetric; falls back to the symmetric curvature when one side has too few points.
- **evaluate_cd0_at_reference_cl(cl_in_fit_window, cd_in_fit_window, cl_at_min_drag, cd_at_min_drag, cd2u, cd2l)** — reads the drag off the fitted bucket at the fixed reference CL = 0.5, used for the REexp fit.
- **parse_one_polar(file_path, reynolds_number)** — the orchestrator: turns one polar file into the full set of QProp aero coefficients (CL0, CL_a, CLmin, CLmax, CD0, CD2u, CD2l, CLCD0), with physical-range gates; returns a failure record when the polar is unusable.

**Reynolds scaling and polar assignment**

- **fit_re_exponent_for_naca(naca_code, re_samples, polar_dir)** — fits REexp, the exponent in `CD0(Re) = CD0_ref · (Re/Re_ref)^REexp`, as the slope of log CD0 (at CL = 0.5) against log Re over all converged Re levels of one NACA code. Falls back to −0.5 (Blasius laminar flat-plate scaling) when fewer than 2 points exist or the fit R² is below the gate; the fitted value is clamped to [−1.5, −0.2].
- **build_re_exponent_table(naca_codes, re_samples, polar_dir, label)** — fits REexp for every unique NACA code and returns the lookup table.
- **find_nearest_cached_re(naca_code, target_re, polar_dir)** — finds the valid cached polar whose Reynolds number is closest to the one a station actually operates at.
- **retrieve_polar_for_station(naca_code, reference_re, station_name, polar_dir, re_exponent_table, s1_station_row)** — returns the best available polar for one blade station through the tier hierarchy: `viscous` (exact cache hit at the station's reference Re), `viscous_near_re` (nearest cached Re for the same code, with REref kept at the station's true Re so QProp's REexp scaling extrapolates the drag), `hub_uses_s1` (hub polar failed entirely, the s1 polar is substituted with the hub Re kept as REref), or None (`failed`). The nearest-Re substitution is needed because the hub and inner stations of these small propellers operate below the Re where XFoil converges reliably.
- **assemble_polar_table(propeller_table, station_table, polar_dir, re_exponent_table, label)** — assigns polars to every station of every configuration and returns the wide output table plus the list of configs with at least one failed station.

**Confidence scoring and reporting**

- **assign_thrust_region(r_over_R)** — labels a station hub / mid / outer from its normalised radius (boundaries at r/R = 0.25 and 0.65).
- **compute_confidence_score_for_config(station_polars_dict, station_radii_dict)** — combines per-station polar quality into one thrust-weighted score in [0, 1]: the hub, mid and outer regions carry 10/40/50 % of the weight (their approximate thrust shares), split equally among the stations in each region; a full viscous polar scores its whole station weight, the hub-uses-s1 fallback scores half, a failed station scores zero.
- **compute_all_confidence_scores(output_table)** — rebuilds the per-station tier and radius information from the output table and returns the confidence score list for all configurations.
- **order_output_columns(output_table)** — arranges the output columns in the documented order: identifiers, per-station metadata block, per-station coefficient block.
- **report_check(condition, description)** — pass/fail reporting helper used by the quality report.


In [ ]:
def make_polar_cache_filename(naca_code, reynolds_number, transition_location):

    xtr_tag = f'{int(round(transition_location * 100)):03d}'
    ncrit_tag = f'{int(ncrit)}'

    return f'naca{str(naca_code).zfill(4)}_re{reynolds_number:06d}_xtr{xtr_tag}_n{ncrit_tag}.txt'


def get_polar_cache_path(polar_dir, naca_code, reynolds_number, transition_location):

    filename = make_polar_cache_filename(naca_code, reynolds_number, transition_location)

    return polar_dir / filename


def polar_cache_is_valid(cache_path):

    return cache_path.exists() and cache_path.stat().st_size >= min_valid_file_bytes


def get_transition_location_for_re(reynolds_number):

    if reynolds_number <= hub_re_threshold:
        transition_location = xtr_hub
    else:
        transition_location = xtr_outer

    return transition_location


def build_chord_or_twist_spline(r_inner, r_mid, r_outer, value_inner, value_mid, value_outer):

    return CubicSpline([r_inner, r_mid, r_outer], [value_inner, value_mid, value_outer], bc_type='natural')


def compute_reference_re(radius_mm, chord_mm, angular_velocity):

    local_velocity = angular_velocity * (radius_mm / 1000.0)
    reynolds_raw = local_velocity * (chord_mm / 1000.0) / air_kinematic_viscosity
    reynolds_clamped = float(np.clip(reynolds_raw, re_floor, re_ceiling))

    return int(round(reynolds_clamped / re_rounding_step) * re_rounding_step)


def make_station_row(config_id, station_name, radius_mm, naca_code, chord_spline, twist_spline):

    chord_mm = float(chord_spline(radius_mm))
    twist_deg = float(twist_spline(radius_mm))
    if chord_mm <= 0:
        raise ValueError(f'Config {config_id} station {station_name}: chord spline gives non-positive chord ({chord_mm:.3f} mm) at r = {radius_mm:.2f} mm. Check inner/mid/outer chord values in the geometry CSV.')
    reference_re = compute_reference_re(radius_mm, chord_mm, launch_omega)

    return {
        'config_id': config_id,
        'station': station_name,
        'naca': str(naca_code).zfill(4),
        'radius_mm': round(radius_mm, 3),
        'chord_mm': round(chord_mm, 4),
        'twist_deg': round(twist_deg, 4),
        'reference_re': reference_re,
    }


def build_station_table(propeller_table):

    station_rows = []
    for row in propeller_table.itertuples(index=False):
        config_id = int(row.config_id)
        tip_radius = float(row.radius_mm)
        mid_radius = float(row.mid_radial_pos) * tip_radius
        chord_spline = build_chord_or_twist_spline(inner_anchor_radius_mm, mid_radius, tip_radius, float(row.inner_chord_mm), float(row.mid_chord_mm), float(row.outer_chord_mm))
        twist_spline = build_chord_or_twist_spline(inner_anchor_radius_mm, mid_radius, tip_radius, float(row.inner_angle_deg), float(row.mid_angle_deg), float(row.outer_angle_deg))
        station_rows.append(make_station_row(config_id, 'hub', hub_station_radius_mm, row.naca_hub, chord_spline, twist_spline))
        for station_name, fraction in zip(qprop_station_names, qprop_station_fractions):
            radius = fraction * tip_radius
            if radius <= hub_station_radius_mm:
                radius = hub_station_radius_mm + 0.1
            naca_code = getattr(row, f'naca_{station_name}')
            station_rows.append(make_station_row(config_id, station_name, radius, naca_code, chord_spline, twist_spline))

    return pd.DataFrame(station_rows)


def derive_re_sample_grid(station_table):

    omega_at_max_rpm = max_rpm * 2 * np.pi / 60.0
    omega_at_floor_rpm = floor_rpm * 2 * np.pi / 60.0

    re_at_max_rpm = []
    re_at_floor_rpm = []
    for row in station_table.itertuples(index=False):
        radius_m = row.radius_mm / 1000.0
        chord_m = row.chord_mm / 1000.0
        re_at_max_rpm.append(omega_at_max_rpm * radius_m * chord_m / air_kinematic_viscosity)
        re_at_floor_rpm.append(omega_at_floor_rpm * radius_m * chord_m / air_kinematic_viscosity)

    re_grid_bottom = re_floor
    re_grid_top = min(re_ceiling, int(max(re_at_max_rpm)))

    num_decades = np.log10(re_grid_top / re_grid_bottom)
    num_samples = max(5, int(np.ceil(num_decades * re_samples_per_decade)))
    raw_grid = np.geomspace(re_grid_bottom, re_grid_top, num_samples)

    rounded_values = set()
    for re_value in raw_grid:
        rounded_values.add(int(round(re_value / re_rounding_step) * re_rounding_step))
    re_samples = []
    for re_value in sorted(rounded_values):
        if re_floor <= re_value <= re_ceiling:
            re_samples.append(re_value)

    below_floor_count = 0
    for re_value in re_at_floor_rpm:
        if re_value < re_floor:
            below_floor_count += 1
    pct_below_floor = 100.0 * below_floor_count / len(re_at_floor_rpm)

    return {
        're_samples': re_samples,
        'num_decades': num_decades,
        're_max_in_data': int(max(re_at_max_rpm)),
        're_min_in_data': int(min(re_at_floor_rpm)),
        'pct_below_floor': pct_below_floor,
    }


def build_job_list(naca_codes, re_samples):

    jobs = []
    for naca_code in naca_codes:
        for reynolds_number in re_samples:
            jobs.append((naca_code, reynolds_number, get_transition_location_for_re(reynolds_number)))

    return jobs

In [ ]:
def build_xfoil_batch_script(naca_code, polar_output_file, reynolds_number, transition_location):

    naca_str = str(naca_code).zfill(4)
    script = (
        f'PLOP\nG F\n\n'
        f'NACA {naca_str}\n'
        f'OPER\n'
        f'ITER {xfoil_max_iterations}\n'
        f'VISC {reynolds_number}\n'
        f'VPAR\n'
        f'N {ncrit}\n'
        f'XTR {transition_location} {transition_location}\n'
        f'\n'
        f'PACC\n{polar_output_file}\n\n'
        f'ASEQ {alpha_start} {alpha_end} {alpha_step}\n'
        f'PACC\n\nQUIT\n'
    )

    return script


def run_one_xfoil_job(polar_dir, naca_code, reynolds_number, transition_location):

    cache_path = get_polar_cache_path(polar_dir, naca_code, reynolds_number, transition_location)
    if polar_cache_is_valid(cache_path):

        return True

    with tempfile.TemporaryDirectory() as work_dir:
        polar_file = Path(work_dir) / 'polar.txt'
        script_text = build_xfoil_batch_script(naca_code, str(polar_file), reynolds_number, transition_location)
        script_file = Path(work_dir) / 'script.txt'
        script_file.write_text(script_text)
        try:
            subprocess.run([str(xfoil_exe)], stdin=script_file.open(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=xfoil_timeout_sec)
        except subprocess.TimeoutExpired:

            return False
        except FileNotFoundError:
            raise FileNotFoundError(f'XFoil executable not found at {xfoil_exe}. Place xfoil.exe in the utils/ folder.')
        if polar_file.exists() and polar_file.stat().st_size >= min_valid_file_bytes:
            polar_file.replace(cache_path)

            return True

    return False


def run_xfoil_jobs_in_parallel(jobs, polar_dir, label='XFoil'):

    total_jobs = len(jobs)
    pending_jobs = []
    for naca_code, reynolds_number, transition_location in jobs:
        cache_path = get_polar_cache_path(polar_dir, naca_code, reynolds_number, transition_location)
        if not polar_cache_is_valid(cache_path):
            pending_jobs.append((naca_code, reynolds_number, transition_location))
    already_done = total_jobs - len(pending_jobs)

    print(f'{label}: {total_jobs} total jobs  |  {already_done} cached  |  {len(pending_jobs)} to run')
    if len(pending_jobs) == 0:
        print('All polars already cached — nothing to run.')

        return

    with ThreadPoolExecutor(max_workers=num_parallel_workers) as executor:
        futures = {}
        for naca_code, reynolds_number, transition_location in pending_jobs:
            future = executor.submit(run_one_xfoil_job, polar_dir, naca_code, reynolds_number, transition_location)
            futures[future] = (naca_code, reynolds_number, transition_location)
        with tqdm(total=len(pending_jobs), desc=label) as progress_bar:
            for future in as_completed(futures):
                progress_bar.update(1)

In [ ]:
def read_xfoil_polar_file(file_path):

    with open(file_path, encoding='utf-8', errors='ignore') as polar_file:
        all_lines = polar_file.readlines()

    data_start_line = None
    for line_index, line in enumerate(all_lines):
        if line.strip().startswith('---') and line_index > 5:
            data_start_line = line_index + 1
            break
    if data_start_line is None:
        raise ValueError(f'No data separator found in {file_path.name}')

    data_rows = []
    for line in all_lines[data_start_line:]:
        stripped = line.strip()
        if not stripped:
            continue
        try:
            values = []
            for value_text in stripped.split():
                values.append(float(value_text))
        except ValueError:
            continue
        if len(values) >= 3:
            data_rows.append(values[:3])
    if len(data_rows) < min_valid_polar_rows:
        raise ValueError(f'{file_path.name}: only {len(data_rows)} data rows — polar likely did not converge')

    data_array = np.array(data_rows, dtype=float)

    return data_array[:, 0], data_array[:, 1], data_array[:, 2]


def remove_convergence_glitches(alpha_values, cl_values, cd_values):

    sort_order = np.argsort(alpha_values)
    alpha_values = alpha_values[sort_order]
    cl_values = cl_values[sort_order]
    cd_values = cd_values[sort_order]

    cl_steps = np.abs(np.diff(cl_values))
    keep_mask = np.concatenate([[True], cl_steps < cl_jump_threshold])

    alpha_values = alpha_values[keep_mask]
    cl_values = cl_values[keep_mask]
    cd_values = cd_values[keep_mask]

    if len(alpha_values) < min_valid_polar_rows:
        raise ValueError('Too few points remaining after removing convergence glitches')

    return alpha_values, cl_values, cd_values


def get_attached_flow_mask(alpha_values):

    primary_mask = (alpha_values >= fit_window_alpha_low) & (alpha_values <= fit_window_alpha_high)
    if primary_mask.sum() >= min_points_for_fit:

        return primary_mask

    wide_mask = (alpha_values >= -2.0) & (alpha_values <= fit_window_alpha_high)
    if wide_mask.sum() >= min_points_for_fit:

        return wide_mask

    raise ValueError(f'Not enough alpha points in the fit window [{fit_window_alpha_low}°, {fit_window_alpha_high}°] — polar may not have converged in the attached-flow regime')


def fit_lift_curve(alpha_in_fit_window, cl_in_fit_window):

    slope_per_deg, intercept = np.polyfit(alpha_in_fit_window, cl_in_fit_window, 1)
    cl0 = float(intercept)
    cla_per_rad = float(slope_per_deg) * (180.0 / np.pi)

    return cl0, cla_per_rad


def fit_drag_polar(cl_in_fit_window, cd_in_fit_window):

    quadratic_coeff, linear_coeff, constant_coeff = np.polyfit(cl_in_fit_window, cd_in_fit_window, 2)
    if quadratic_coeff > 1e-10:
        cl_at_min_drag = float(-linear_coeff / (2 * quadratic_coeff))
        cd_at_min_drag = max(float(constant_coeff - linear_coeff ** 2 / (4 * quadratic_coeff)), 1e-5)
        curvature = float(quadratic_coeff)
    else:
        min_cd_index = np.argmin(cd_in_fit_window)
        cl_at_min_drag = float(cl_in_fit_window[min_cd_index])
        cd_at_min_drag = float(cd_in_fit_window[min_cd_index])
        curvature = 0.01

    return cd_at_min_drag, cl_at_min_drag, curvature


def fit_asymmetric_drag_curvatures(cl_in_fit_window, cd_in_fit_window, cl_at_min_drag, symmetric_curvature):

    cl_shifted_squared = (cl_in_fit_window - cl_at_min_drag) ** 2
    upper_mask = cl_in_fit_window >= cl_at_min_drag
    lower_mask = cl_in_fit_window < cl_at_min_drag
    warning_message = None

    def fit_single_curvature(mask):

        x_sq = cl_shifted_squared[mask]
        y = cd_in_fit_window[mask]
        numerator = float(np.dot(y, x_sq))
        denominator = float(np.dot(x_sq, x_sq))
        if denominator < 1e-12:
            curvature_value = symmetric_curvature
        else:
            curvature_value = max(numerator / denominator, 1e-6)

        return curvature_value

    if upper_mask.sum() >= 2:
        cd2u = fit_single_curvature(upper_mask)
    else:
        cd2u = symmetric_curvature
        warning_message = f'CD2u fallback: only {upper_mask.sum()} points above CLCD0 = {cl_at_min_drag:.3f} in the fit window. Using symmetric curvature = {symmetric_curvature:.5f}. This typically happens for highly cambered sections where CLCD0 is near or above the upper fit boundary ({fit_window_alpha_high}°).'

    if lower_mask.sum() >= 2:
        cd2l = fit_single_curvature(lower_mask)
    else:
        cd2l = symmetric_curvature
        lower_warn = f'CD2l fallback: only {lower_mask.sum()} points below CLCD0 = {cl_at_min_drag:.3f} in the fit window. Using symmetric curvature = {symmetric_curvature:.5f}.'
        if warning_message is None:
            warning_message = lower_warn
        else:
            warning_message = warning_message + ' | ' + lower_warn

    return cd2u, cd2l, warning_message


def evaluate_cd0_at_reference_cl(cl_in_fit_window, cd_in_fit_window, cl_at_min_drag, cd_at_min_drag, cd2u, cd2l):

    cl_min_in_window = float(cl_in_fit_window.min())
    cl_max_in_window = float(cl_in_fit_window.max())
    if not (cl_min_in_window <= cd0_reference_cl <= cl_max_in_window):

        return float(np.clip(cd_at_min_drag, cd0_minimum, cd0_maximum))

    delta_cl = cd0_reference_cl - cl_at_min_drag
    if delta_cl >= 0:
        cd_at_ref_cl = cd_at_min_drag + cd2u * delta_cl ** 2
    else:
        cd_at_ref_cl = cd_at_min_drag + cd2l * delta_cl ** 2

    return float(np.clip(cd_at_ref_cl, cd0_minimum, cd0_maximum))


def parse_one_polar(file_path, reynolds_number):

    try:
        alpha_values, cl_values, cd_values = read_xfoil_polar_file(file_path)

        physical_drag_mask = cd_values > 1e-6
        alpha_values = alpha_values[physical_drag_mask]
        cl_values = cl_values[physical_drag_mask]
        cd_values = cd_values[physical_drag_mask]
        if len(alpha_values) < min_valid_polar_rows:
            raise ValueError('Too few rows with positive drag values')

        alpha_values, cl_values, cd_values = remove_convergence_glitches(alpha_values, cl_values, cd_values)
        attached_flow_mask = get_attached_flow_mask(alpha_values)

        cl_in_window = cl_values[attached_flow_mask]
        cd_in_window = cd_values[attached_flow_mask]

        cl0, cla_per_rad = fit_lift_curve(alpha_values[attached_flow_mask], cl_in_window)
        if not (cla_minimum <= cla_per_rad <= cla_maximum):
            raise ValueError(f'CL_a = {cla_per_rad:.3f} rad⁻¹ is outside the physical range [{cla_minimum}, {cla_maximum}]. Polar may not have converged.')

        cd_at_min_drag, cl_at_min_drag, symmetric_curvature = fit_drag_polar(cl_in_window, cd_in_window)
        if not (cd0_minimum <= cd_at_min_drag <= cd0_maximum):
            raise ValueError(f'CD0 = {cd_at_min_drag:.5f} is outside the physical range [{cd0_minimum}, {cd0_maximum}]. Polar may be unphysical at this Re.')

        cd2u, cd2l, curvature_warning = fit_asymmetric_drag_curvatures(cl_in_window, cd_in_window, cl_at_min_drag, symmetric_curvature)
        cd0_for_re_fit = evaluate_cd0_at_reference_cl(cl_in_window, cd_in_window, cl_at_min_drag, cd_at_min_drag, cd2u, cd2l)

        cl_max = float(cl_values.max()) * (1.0 - stall_safety_buffer)
        cl_min = float(cl_values.min()) * (1.0 - stall_safety_buffer)

        return {
            'CL0': round(cl0, 4),
            'CL_a': round(cla_per_rad, 4),
            'CLmin': round(cl_min, 4),
            'CLmax': round(cl_max, 4),
            'CD0': round(cd_at_min_drag, 6),
            'CD2u': round(cd2u, 6),
            'CD2l': round(cd2l, 6),
            'CLCD0': round(cl_at_min_drag, 4),
            'REref': reynolds_number,
            'REexp': None,
            'xfoil_ok': True,
            'cd0_for_re_fit': round(cd0_for_re_fit, 6),
            'curvature_warning': curvature_warning,
        }
    except Exception as error:

        return {
            'parse_error': str(error),
            'xfoil_ok': False,
        }

In [ ]:
def fit_re_exponent_for_naca(naca_code, re_samples, polar_dir):

    re_values_used = []
    cd0_values_used = []
    for reynolds_number in re_samples:
        transition_loc = get_transition_location_for_re(reynolds_number)
        cache_path = get_polar_cache_path(polar_dir, naca_code, reynolds_number, transition_loc)
        if not polar_cache_is_valid(cache_path):
            continue
        polar_result = parse_one_polar(cache_path, reynolds_number)
        if polar_result.get('xfoil_ok') and polar_result.get('cd0_for_re_fit') is not None:
            re_values_used.append(float(reynolds_number))
            cd0_values_used.append(float(polar_result['cd0_for_re_fit']))

    if len(re_values_used) < 2:

        return -0.5, len(re_values_used), None

    log_re = np.log(np.array(re_values_used))
    log_cd0 = np.log(np.array(cd0_values_used))
    try:
        slope, intercept = np.polyfit(log_re, log_cd0, 1)
        fitted_re_exp = float(slope)
        predicted_log_cd0 = slope * log_re + intercept
        residual_sum = np.sum((log_cd0 - predicted_log_cd0) ** 2)
        total_sum = np.sum((log_cd0 - log_cd0.mean()) ** 2)
        if total_sum > 1e-12:
            r_squared = float(1.0 - residual_sum / total_sum)
        else:
            r_squared = 1.0
        if r_squared < re_exponent_r2_gate:

            return -0.5, len(re_values_used), round(r_squared, 3)

        clamped_re_exp = float(np.clip(fitted_re_exp, -1.5, -0.2))

        return clamped_re_exp, len(re_values_used), round(r_squared, 3)
    except Exception:

        return -0.5, len(re_values_used), None


def build_re_exponent_table(naca_codes, re_samples, polar_dir, label='Fitting REexp'):

    re_exponent_table = {}
    for naca_code in tqdm(naca_codes, desc=label):
        re_exp, num_points, r_sq = fit_re_exponent_for_naca(naca_code, re_samples, polar_dir)
        re_exponent_table[naca_code] = {
            'REexp': re_exp,
            'num_re_points': num_points,
            're_fit_r2': r_sq,
        }

    return re_exponent_table


def find_nearest_cached_re(naca_code, target_re, polar_dir):

    naca_prefix = f'naca{str(naca_code).zfill(4)}_'
    best_re = None
    best_gap = float('inf')
    for cache_file in polar_dir.glob(f'{naca_prefix}*.txt'):
        if not polar_cache_is_valid(cache_file):
            continue
        cached_re = None
        for part in cache_file.stem.split('_'):
            if part.startswith('re') and part[2:].isdigit():
                cached_re = int(part[2:])
                break
        if cached_re is None:
            continue
        gap = abs(cached_re - target_re)
        if gap < best_gap:
            best_gap = gap
            best_re = cached_re

    if best_re is None:

        return None, None

    return best_re, get_transition_location_for_re(best_re)


def retrieve_polar_for_station(naca_code, reference_re, station_name, polar_dir, re_exponent_table, s1_station_row=None):

    exact_xtr = get_transition_location_for_re(reference_re)
    exact_cache_path = get_polar_cache_path(polar_dir, naca_code, reference_re, exact_xtr)
    if polar_cache_is_valid(exact_cache_path):
        polar = parse_one_polar(exact_cache_path, reference_re)
        if polar.get('xfoil_ok'):
            polar['REexp'] = re_exponent_table.get(naca_code, {}).get('REexp', -0.5)
            polar['tier'] = 'viscous'

            return polar

    nearest_re, nearest_xtr = find_nearest_cached_re(naca_code, reference_re, polar_dir)
    if nearest_re is not None and nearest_re != reference_re:
        nearest_cache_path = get_polar_cache_path(polar_dir, naca_code, nearest_re, nearest_xtr)
        if polar_cache_is_valid(nearest_cache_path):
            polar = parse_one_polar(nearest_cache_path, nearest_re)
            if polar.get('xfoil_ok'):
                polar['REexp'] = re_exponent_table.get(naca_code, {}).get('REexp', -0.5)
                polar['REref'] = reference_re
                polar['tier'] = 'viscous_near_re'
                polar['cached_re_used'] = nearest_re

                return polar

    if station_name == 'hub' and s1_station_row is not None:
        s1_naca = s1_station_row['naca']
        s1_reference_re = s1_station_row['reference_re']
        s1_xtr = get_transition_location_for_re(s1_reference_re)
        s1_cache_path = get_polar_cache_path(polar_dir, s1_naca, s1_reference_re, s1_xtr)
        if not polar_cache_is_valid(s1_cache_path):
            s1_nearest_re, s1_nearest_xtr = find_nearest_cached_re(s1_naca, s1_reference_re, polar_dir)
            if s1_nearest_re is not None:
                s1_cache_path = get_polar_cache_path(polar_dir, s1_naca, s1_nearest_re, s1_nearest_xtr)
                s1_reference_re = s1_nearest_re
        if polar_cache_is_valid(s1_cache_path):
            polar = parse_one_polar(s1_cache_path, s1_reference_re)
            if polar.get('xfoil_ok'):
                polar['REexp'] = re_exponent_table.get(s1_naca, {}).get('REexp', -0.5)
                polar['REref'] = reference_re
                polar['tier'] = 'hub_uses_s1'

                return polar

    return None


def assemble_polar_table(propeller_table, station_table, polar_dir, re_exponent_table, label='Assembling'):

    output_rows = []
    failed_config_ids = []
    for config_id in tqdm(propeller_table['config_id'], desc=label):
        config_id = int(config_id)
        this_prop = station_table[station_table['config_id'] == config_id]

        output_row = {'config_id': config_id}
        config_is_complete = True

        s1_rows = this_prop[this_prop['station'] == 's1']
        if not s1_rows.empty:
            s1_row = s1_rows.iloc[0].to_dict()
        else:
            s1_row = None

        for station_name in all_station_names:
            station_rows = this_prop[this_prop['station'] == station_name]
            if station_rows.empty:
                config_is_complete = False
                continue
            station_row = station_rows.iloc[0]

            if station_name == 'hub':
                s1_row_for_fallback = s1_row
            else:
                s1_row_for_fallback = None
            polar = retrieve_polar_for_station(station_row['naca'], station_row['reference_re'], station_name, polar_dir, re_exponent_table, s1_station_row=s1_row_for_fallback)

            output_row[f'naca_{station_name}'] = station_row['naca']
            output_row[f're_{station_name}'] = station_row['reference_re']
            output_row[f'r_{station_name}_mm'] = round(station_row['radius_mm'], 3)
            output_row[f'chord_{station_name}_mm'] = round(station_row['chord_mm'], 3)
            output_row[f'twist_{station_name}_deg'] = round(station_row['twist_deg'], 3)
            if polar:
                output_row[f'tier_{station_name}'] = polar['tier']
            else:
                output_row[f'tier_{station_name}'] = 'failed'
            if polar and polar.get('tier') == 'viscous_near_re':
                output_row[f'cached_re_{station_name}'] = polar.get('cached_re_used')
            else:
                output_row[f'cached_re_{station_name}'] = None
            output_row[f're_exp_r2_{station_name}'] = re_exponent_table.get(station_row['naca'], {}).get('re_fit_r2')

            if polar:
                for coeff_name in qprop_aero_keys:
                    output_row[f'{station_name}_{coeff_name}'] = polar.get(coeff_name)
                if polar.get('curvature_warning'):
                    warnings.warn(f'Config {config_id} station {station_name}: ' + polar['curvature_warning'])
            else:
                for coeff_name in qprop_aero_keys:
                    output_row[f'{station_name}_{coeff_name}'] = None
                config_is_complete = False

        output_rows.append(output_row)
        if not config_is_complete:
            failed_config_ids.append(config_id)

    return pd.DataFrame(output_rows), failed_config_ids

In [ ]:
def assign_thrust_region(r_over_R):

    if r_over_R < 0.25:
        region = 'hub'
    elif r_over_R < 0.65:
        region = 'mid'
    else:
        region = 'outer'

    return region


def compute_confidence_score_for_config(station_polars_dict, station_radii_dict):

    if station_radii_dict:
        tip_radius = max(station_radii_dict.values())
    else:
        tip_radius = 1.0

    region_weights = {
        'hub': 0.10,
        'mid': 0.40,
        'outer': 0.50,
    }

    stations_per_region = {'hub': 0, 'mid': 0, 'outer': 0}
    for station_name in station_polars_dict:
        radius_mm = station_radii_dict.get(station_name, 0.0)
        region = assign_thrust_region(radius_mm / tip_radius)
        stations_per_region[region] += 1

    total_score = 0.0
    for station_name, polar in station_polars_dict.items():
        radius_mm = station_radii_dict.get(station_name, 0.0)
        region = assign_thrust_region(radius_mm / tip_radius)
        num_in_region = stations_per_region[region]
        if num_in_region == 0:
            continue
        weight_per_station = region_weights[region] / num_in_region
        if polar is None:
            contribution = 0.0
        elif polar.get('tier') == 'hub_uses_s1':
            contribution = 0.5 * weight_per_station
        else:
            contribution = weight_per_station
        total_score += contribution

    return round(total_score, 4)


def compute_all_confidence_scores(output_table):

    confidence_scores = []
    for config_id in output_table['config_id']:
        config_id = int(config_id)
        polars_for_score = {}
        radii_for_score = {}
        for station_name in all_station_names:
            tier_col = f'tier_{station_name}'
            r_col = f'r_{station_name}_mm'
            if tier_col in output_table.columns:
                tier = output_table.loc[output_table['config_id'] == config_id, tier_col].values[0]
                if r_col in output_table.columns:
                    radii_for_score[station_name] = float(output_table.loc[output_table['config_id'] == config_id, r_col].values[0])
                else:
                    radii_for_score[station_name] = 0.0
                if pd.isna(tier) or tier == 'failed':
                    polars_for_score[station_name] = None
                else:
                    polars_for_score[station_name] = {'tier': tier}
        score = compute_confidence_score_for_config(polars_for_score, radii_for_score)
        confidence_scores.append(score)

    return confidence_scores


def order_output_columns(output_table):

    identifier_columns = ['config_id', 'confidence_score']
    metadata_columns = []
    coefficient_columns = []
    for station_name in all_station_names:
        metadata_columns.append(f'naca_{station_name}')
        metadata_columns.append(f're_{station_name}')
        metadata_columns.append(f'r_{station_name}_mm')
        metadata_columns.append(f'chord_{station_name}_mm')
        metadata_columns.append(f'twist_{station_name}_deg')
        metadata_columns.append(f'tier_{station_name}')
        metadata_columns.append(f'cached_re_{station_name}')
        metadata_columns.append(f're_exp_r2_{station_name}')
        for key in qprop_aero_keys:
            coefficient_columns.append(f'{station_name}_{key}')

    all_output_columns = []
    for column in identifier_columns + metadata_columns + coefficient_columns:
        if column in output_table.columns:
            all_output_columns.append(column)

    return output_table[all_output_columns].sort_values('config_id').reset_index(drop=True)


def report_check(condition, description):

    global all_checks_passed
    status = 'OK  ' if condition else 'FAIL'
    print(f'  [{status}]  {description}')
    if not condition:
        all_checks_passed = False

## 4. Main Code

The main code loads the configuration and the NB1/NB3 inputs, builds the per-station geometry table, derives the Reynolds sample grid from the actual geometry and RPM envelope, runs the XFoil sweep (cached), fits the Reynolds scaling exponent per NACA code, assembles and saves the polar table with confidence scores, repeats the identical flow for the 10 recovered validation propellers (in a separate polar cache, `xfoil_polars/validation/`, so the subset stays self-contained and reproducible), and closes with a quality report.


### 4.1 Load Input Data

In [ ]:
naca_column_names = ['naca_hub', 'naca_s1', 'naca_s2', 'naca_s3', 'naca_s4', 'naca_s5', 'naca_s6']
naca_dtype = {}
for column in naca_column_names:
    naca_dtype[column] = str

naca_data = pd.read_csv(naca_csv, dtype=naca_dtype)
for column in naca_column_names:
    naca_data[column] = naca_data[column].str.zfill(4)

geometry_data = pd.read_csv(geometry_csv)

propeller_data = pd.merge(geometry_data, naca_data, on='config_id', how='inner', validate='one_to_one').sort_values('config_id').reset_index(drop=True)

print(f'Loaded {len(propeller_data)} propeller configurations.')
print(f'  Geometry : {geometry_csv.name}')
print(f'  NACA     : {naca_csv.name}')
propeller_data.head(2)

### 4.2 Build the Per-Station Geometry Table

Each propeller receives 7 aerodynamic stations: the hub station fixed at 8.25 mm (the QProp root station inside the hub body) and s1–s6 at r/R fractions [0.20, 0.35, 0.50, 0.65, 0.80, 0.92]. Chord and twist at each station are reconstructed with the same natural cubic spline through the inner/mid/outer anchors that Rhino uses for the blade loft, and every station gets its reference Reynolds number at launch RPM.


In [ ]:
station_table = build_station_table(propeller_data)
all_naca_codes = station_table['naca'].unique()

print(f'Station table: {len(station_table)} rows  ({len(propeller_data)} propellers × 7 stations)')
print(f'Unique NACA codes to simulate: {len(all_naca_codes)}')
print()
print('Reference Reynolds number by station (at launch RPM, static hover):')
for station in all_station_names:
    re_values = station_table[station_table['station'] == station]['reference_re']
    print(f'  {station:4s}:  min = {re_values.min():6,}   median = {int(re_values.median()):6,}   max = {re_values.max():6,}')

### 4.3 Derive the Reynolds Number Sample Grid

The sample grid is derived from the actual station geometry across the full RPM envelope rather than hardcoded, so it adapts automatically when the launch RPM changes. Station–RPM combinations below the XFoil floor are handled downstream by the REexp power-law extrapolation, a documented limitation of the approach.


In [ ]:
grid_info = derive_re_sample_grid(station_table)
re_samples = grid_info['re_samples']

print(f'RPM envelope          : {floor_rpm}–{max_rpm} RPM')
print(f'Re range in dataset   : {grid_info["re_min_in_data"]:,} – {grid_info["re_max_in_data"]:,}')
print(f'Re sample grid        : {re_samples}')
print(f'  → {len(re_samples)} grid points across {grid_info["num_decades"]:.2f} decades of Re')
print()
print(f'Station-RPM combos below re_floor ({re_floor:,}): {grid_info["pct_below_floor"]:.1f}%')
print('  These occur at the very lowest RPMs where thrust is negligible.')
print('  The REexp power law is extrapolated below re_floor in those cases.')
print()
print('If you change the launch RPM, re-run from this cell — the grid updates automatically.')

### 4.4 Run the XFoil Sweep

Every unique NACA code is simulated at every Re sample level. Cached polars are reused, so re-running this cell is cheap.


In [ ]:
all_jobs = build_job_list(all_naca_codes, re_samples)

print(f'Unique NACA codes : {len(all_naca_codes)}')
print(f'Re sample levels  : {len(re_samples)}')
print(f'Total XFoil jobs  : {len(all_jobs)}  ({len(all_naca_codes)} codes × {len(re_samples)} Re values)')
print()

start_time = time.time()
run_xfoil_jobs_in_parallel(all_jobs, polar_dir_main, label='XFoil multi-Re sweep')
print(f'Completed in {time.time() - start_time:.1f} seconds')

### 4.5 Fit the Reynolds Scaling Exponent

REexp tells QProp how drag changes when the propeller spins away from the reference RPM. The expected physical range is −0.4 to −0.8: −0.5 is the Blasius laminar flat-plate scaling and −0.2 the Prandtl–Kármán turbulent flat-plate scaling; low-Re propeller blades sit between these extremes.


In [ ]:
print('Fitting Re-scaling exponent for every unique NACA code...')
re_exponent_table = build_re_exponent_table(all_naca_codes, re_samples, polar_dir_main)

num_fitted = 0
num_poor_fit = 0
num_no_data = 0
all_exponents = []
for entry in re_exponent_table.values():
    all_exponents.append(entry['REexp'])
    if entry['num_re_points'] < 2:
        num_no_data += 1
    elif entry['re_fit_r2'] is not None and entry['re_fit_r2'] >= re_exponent_r2_gate:
        num_fitted += 1
    else:
        num_poor_fit += 1

print()
print(f'Fitted successfully (R² ≥ {re_exponent_r2_gate}) : {num_fitted} NACA codes')
print(f'Poor fit → defaulted to −0.5               : {num_poor_fit} codes')
print(f'Too few Re points → defaulted to −0.5      : {num_no_data} codes')
print(f'REexp range   : {min(all_exponents):.3f} to {max(all_exponents):.3f}')
print(f'REexp median  : {float(np.median(all_exponents)):.3f}')
print()
print('Expected physical range: −0.4 to −0.8')
print('  −0.5 = Blasius laminar flat-plate scaling')
print('  −0.2 = Prandtl–Kármán turbulent flat-plate scaling')
print('Values below −1.0 indicate a strong laminar drag rise — review those polars.')

### 4.6 Assemble the Polar Table and Save

Assigns the best available polar to every station of every configuration through the tier hierarchy, adds the thrust-weighted confidence score, orders the columns (identifiers → per-station metadata → per-station coefficients) and saves the outputs. `cached_re_<station>` is only populated when the tier is `viscous_near_re` and records how far the used polar was from the station's target Re.


In [ ]:
print('Assembling polar table for all propeller configurations...')
output_table, failed_config_ids = assemble_polar_table(propeller_data, station_table, polar_dir_main, re_exponent_table)

print()
if failed_config_ids:
    print(f'WARNING: {len(failed_config_ids)} configurations have at least one failed station.')
else:
    print('All configurations assembled successfully.')

print('Computing confidence scores...')
confidence_scores = compute_all_confidence_scores(output_table)
output_table.insert(1, 'confidence_score', confidence_scores)

score_series = output_table['confidence_score']
print('Confidence score summary:')
print(f'  Min    : {score_series.min():.3f}')
print(f'  Median : {score_series.median():.3f}')
print(f'  Mean   : {score_series.mean():.3f}')
print(f'  Max    : {score_series.max():.3f}')
print()
pct_above_half = (score_series >= 0.5).mean() * 100
print(f'  {pct_above_half:.1f}% of configurations have confidence ≥ 0.5')
print()
print('Configurations with score < 0.5 should be excluded from the QProp simulation.')

output_table = order_output_columns(output_table)
output_table.to_csv(output_csv, index=False)

print(f'Saved -> {output_csv}')
print(f'  {len(output_table)} rows  x  {len(output_table.columns)} columns')

if failed_config_ids:
    pd.DataFrame({'config_id': failed_config_ids}).to_csv(failed_csv_path, index=False)
    print(f'Failed configs -> {failed_csv_path.name}  ({len(failed_config_ids)} entries)')

output_table.head(3)

### 4.7 Validation Subset — Recovered Geometry

Repeats the identical flow for the 10 recovered validation propellers from `utils/00_validation_geometry.csv` (see NB3). The subset uses its own polar cache (`xfoil_polars/validation/`) and derives its own Re grid from its own station geometry, so its outputs are self-contained and reproducible independently of the main design space.


In [ ]:
val_geometry_path = measurements.validation_geometry_path(base_dir)
val_naca_csv = csv_dir / 'val_03_naca_codes.csv'

if val_geometry_path.exists() and val_naca_csv.exists():
    val_geometry_data = measurements.load_recovered_validation_geometry(base_dir)
    val_naca_data = pd.read_csv(val_naca_csv, dtype=naca_dtype)
    for column in naca_column_names:
        val_naca_data[column] = val_naca_data[column].str.zfill(4)
    val_propeller_data = pd.merge(val_geometry_data, val_naca_data, on='config_id', how='inner', validate='one_to_one').sort_values('config_id').reset_index(drop=True)
    print(f'Validation subset: {len(val_propeller_data)} propellers -> {sorted(val_propeller_data["config_id"].tolist())}')

    val_station_table = build_station_table(val_propeller_data)
    val_naca_codes = val_station_table['naca'].unique()
    val_grid_info = derive_re_sample_grid(val_station_table)
    val_re_samples = val_grid_info['re_samples']
    print(f'Validation Re grid: {val_re_samples}')

    val_jobs = build_job_list(val_naca_codes, val_re_samples)
    run_xfoil_jobs_in_parallel(val_jobs, validation_polar_dir, label='XFoil validation subset')

    val_re_exponent_table = build_re_exponent_table(val_naca_codes, val_re_samples, validation_polar_dir, label='Fitting REexp (validation)')

    val_output_table, val_failed_ids = assemble_polar_table(val_propeller_data, val_station_table, validation_polar_dir, val_re_exponent_table, label='Assembling validation subset')
    val_confidence_scores = compute_all_confidence_scores(val_output_table)
    val_output_table.insert(1, 'confidence_score', val_confidence_scores)
    val_output_table = order_output_columns(val_output_table)
    val_output_table.to_csv(csv_dir / 'val_04_xfoil_polars.csv', index=False)
    print(f'Saved -> val_04_xfoil_polars.csv  ({len(val_output_table)} rows)')

    if val_failed_ids:
        pd.DataFrame({'config_id': val_failed_ids}).to_csv(csv_dir / 'val_04_xfoil_failed_configs.csv', index=False)
        print(f'Failed validation configs -> val_04_xfoil_failed_configs.csv  ({len(val_failed_ids)} entries)')
else:
    print('Validation geometry or val_03_naca_codes.csv not found — skipping the validation subset.')
    print('Run NB3 first to generate the validation-subset inputs.')

### 4.8 Validation and Quality Report

Sanity checks on the main output before handing it to Notebook 5. A failing check here is much cheaper to investigate than a wrong QProp result.


In [ ]:
print('=' * 65)
print('VALIDATION REPORT')
print('=' * 65)

all_checks_passed = True
num_propellers = len(propeller_data)

report_check(len(output_table) == num_propellers, f'Row count matches input ({num_propellers} propellers)')
report_check(output_table['config_id'].is_unique, 'Each config_id appears exactly once')

print()
print('Station coverage (failure rate must be < 5%):')
for station_name in all_station_names:
    tier_column = f'tier_{station_name}'
    num_viscous = (output_table[tier_column] == 'viscous').sum()
    num_near_re = (output_table[tier_column] == 'viscous_near_re').sum()
    num_hub_fallbk = (output_table[tier_column] == 'hub_uses_s1').sum()
    num_failed = (output_table[tier_column] == 'failed').sum()
    failure_rate = num_failed / num_propellers
    report_check(failure_rate < 0.05, f'{station_name:4s}  viscous={num_viscous}  near_re={num_near_re}  hub_s1={num_hub_fallbk}  failed={num_failed}  ({100 * failure_rate:.1f}%)')

print()
print('Nearest-Re substitution diagnostic (viscous_near_re stations):')
for station_name in all_station_names:
    tier_col = f'tier_{station_name}'
    cached_re_col = f'cached_re_{station_name}'
    re_col = f're_{station_name}'
    if cached_re_col not in output_table.columns:
        continue
    near_re_mask = output_table[tier_col] == 'viscous_near_re'
    num_near = near_re_mask.sum()
    if num_near == 0:
        continue
    target_re_vals = output_table.loc[near_re_mask, re_col].astype(float)
    cached_re_vals = output_table.loc[near_re_mask, cached_re_col].astype(float)
    re_ratio_vals = cached_re_vals / target_re_vals
    print(f'  {station_name:4s}: {num_near} stations used nearest-Re polar')
    print(f'        target Re range : {int(target_re_vals.min()):,} - {int(target_re_vals.max()):,}')
    print(f'        cached Re range : {int(cached_re_vals.min()):,} - {int(cached_re_vals.max()):,}')
    print(f'        Re ratio (cached/target): median={re_ratio_vals.median():.1f}x  max={re_ratio_vals.max():.1f}x')
    print(f'        (REexp power law extrapolates drag from cached Re down to target Re)')

print()
print('Confidence score:')
score_values = output_table['confidence_score']
print(f'  min={score_values.min():.3f}  median={score_values.median():.3f}  mean={score_values.mean():.3f}  max={score_values.max():.3f}')
report_check(score_values.between(0.0, 1.0).all(), 'All confidence scores in [0, 1]')
report_check((score_values >= 0.5).mean() > 0.90, '> 90% of configurations have confidence score >= 0.5')

print()
print('REexp fit quality (fraction of codes with R2 above gate):')
for station_name in all_station_names:
    r2_column = f're_exp_r2_{station_name}'
    if r2_column in output_table.columns:
        r2_values = output_table[r2_column].dropna()
        if len(r2_values) > 0:
            pct_good = (r2_values >= re_exponent_r2_gate).mean() * 100
            print(f'  {station_name:4s}: {pct_good:.1f}% of fitted codes have R2 >= {re_exponent_r2_gate}')

print()
print('CD0 physical range check:')
for station_name in all_station_names:
    cd0_column = f'{station_name}_CD0'
    cd0_values = output_table[cd0_column].dropna()
    if len(cd0_values) > 0:
        report_check(cd0_values.between(cd0_minimum, cd0_maximum).all(), f'{station_name}: all CD0 in [{cd0_minimum}, {cd0_maximum}]  (min={cd0_values.min():.4f}  max={cd0_values.max():.4f})')

print()
print('=' * 65)
if all_checks_passed:
    print('ALL CHECKS PASSED -- output is ready for Notebook 5.')
else:
    print('SOME CHECKS FAILED -- review the output above before proceeding.')
print('=' * 65)